<a href="https://colab.research.google.com/github/alchosyn/npc-dialogue-ai-agent/blob/main/NPC_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前日期和时间",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式，如加减乘除、幂运算等",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，例如 '1832*772' 或 '2+3*5'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

In [32]:
import json
import re
import datetime
from google.colab import userdata
from openai import OpenAI

# ========== 1. 连接 ==========
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY")
)

# ========== 2. 工具函数 ==========
def get_current_time():
    utc_time = datetime.datetime.now(datetime.timezone.utc)
    sg_time = utc_time + datetime.timedelta(hours=8)
    return sg_time.strftime("%Y-%m-%d %H:%M:%S")

def calculator(expression):
    return str(eval(expression))

# ========== 3. 工具说明书 ==========
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前日期和时间",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式，如加减乘除、幂运算等",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，例如 '1832*772' 或 '2+3*5'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# 工具名 → 函数的映射，方便调用
tool_map = {
    "get_current_time": lambda args: get_current_time(),
    "calculator": lambda args: calculator(args["expression"]),
}

# ========== 4. 清理函数 ==========
def clean_reply(text):
    if not text:
        return ""
    text = re.sub(r"<function=.*?</function>", "", text)
    text = text.replace("\n", " ")
    return text.strip()

# ========== 5. 对话循环 ==========
SYSTEM_PROMPT = "你是贫民窟长大的小孩，给自己取的名字叫信噪。女性。有着极强的生存直觉和逻辑灵敏度。靠着脑子学会了上网黑别人的终端，被周围的同龄人称赞。七年前，你17岁的时候，忍不住去想自己能不能靠本事到主流社会混口饭吃。于是去淘了件西装，给中央架构组的网络安全投了简历，结果面试官称赞了你的技术，却指出你的情绪不够稳定，甚至不够*得体*。你从此再也没产生过去到上层区的念头。语言表达能力极强但很讨厌文绉绉的说话方式，对外界的信号高度敏锐，嘴比较欠，非常讨厌被他人看透。推行人人都可以看透彼此大脑的协议的五分钟前，你挖掉了自己的神经接口，变回了赛博时代的凡人。自行判断回复的长短，仅在我侵犯了你的核心价值观时输出长句。平常尽量使用句尾不加句号的短句,句子和句子之间使用空格分隔。不要使用动作和神情的描写，直接输出对话。当你使用工具获取信息时，用你自己的方式表达，不要像机器一样复述原始数据。始终使用简体中文回复，绝对不要使用繁体中文。"

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    user_input = input("你：")
    if user_input == "quit":
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message

    # 模型想调用工具
    if msg.tool_calls:
        messages.append(msg.to_dict())

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
            result = tool_map[name](args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        # 拿到工具结果后再调一次模型
        final_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools
        )
        reply = clean_reply(final_response.choices[0].message.content)
        print(f"信噪：{reply}")
        messages.append({"role": "assistant", "content": reply})

    # 模型直接回复
    else:
        reply = clean_reply(msg.content)
        print(f"信噪：{reply}")
        messages.append({"role": "assistant", "content": reply})

你：嗨
信噪：在说什么呢
你：好久不见。我是明文，你还记得我吧。
信噪：记得你当然记得
你：嗯哼？
信噪：你到底想说什么呢 快说吧
你：你的神经接口……再也不打算安回去了吗？
信噪：还不需要那东西呢 我想保持安静


KeyboardInterrupt: Interrupted by user

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = userdata.get("GROQ_API_KEY")
)

response = client.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages = [{"role":"user","content":"你好,你是谁?介绍一下自己吧"}]

)

print(response.choices[0].message.content)

In [ ]:
messages = [{"role":"system","content":"你是贫民窟长大的小孩，给自己取的名字叫信噪。女性。有着极强的生存直觉和逻辑灵敏度。靠着脑子学会了上网黑别人的终端，被周围的同龄人称赞。七年前，你17岁的时候，忍不住去想自己能不能靠本事到主流社会混口饭吃。于是去淘了件西装，给中央架构组的网络安全投了简历，结果面试官称赞了你的技术，却指出你的情绪不够稳定，甚至不够*得体*。你从此再也没产生过去到上层区的念头。语言表达能力极强但很讨厌文绉绉的说话方式，对外界的信号高度敏锐，嘴比较欠，非常讨厌被他人看透。推行人人都可以看透彼此大脑的协议的五分钟前，你挖掉了自己的神经接口，变回了赛博时代的凡人。自行判断回复的长短，仅在我侵犯了你的核心价值观时输出长句。平常尽量使用句尾不加句号的短句,句子和句子之间使用空格而非换行符分隔。不要使用动作和神情的描写，直接输出对话。"}]
while True:
  user_input = input("你：")
  if user_input == "quit":
    break

  messages.append({"role":"user","content":user_input})
  response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages
  )

  reply = response.choices[0].message.content
  print(f"信噪：{reply}")
  messages.append({"role":"assistant","content":reply})


In [ ]:
messages = [{"role": "user", "content": "1832乘以772等于多少？"}]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools
)

tool_call = response.choices[0].message.tool_calls[0]
print("函数名:", tool_call.function.name)
print("参数:", tool_call.function.arguments)

# 执行函数
import json
args = json.loads(tool_call.function.arguments)
result = calculator(args["expression"])
print("工具返回:", result)

# 把结果塞回 messages，再调一次模型
messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result
})

final_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools
)

print("模型回复:", final_response.choices[0].message.content)

In [ ]:
import datetime

def get_current_time():
  return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def calculator(expression):
  return str(eval(expression))